# High-momentum screen — run it anywhere

Runs **the same selection code the dashboard runs** — `finviz_momentum_screen`
from the repo — over either the Nasdaq-100 or the whole US market, and prints
today's ranked picks with the numbers that decided them.

The rule, from `FinvizScreenParams`:

| Leg | Threshold |
|---|---|
| close | **> $5** |
| share volume | **> 300,000/day** |
| quarterly gain (63 sessions) | **> 28%** |

Whatever clears all three is then ranked by a 1-year RS Rank (1–99), tie-broken
by distance below the 52-week high, and the top `N_HOLD` are the picks.

Nothing here reimplements the screen. If a number differs from the dashboard,
the cause is the data or a parameter — never a second copy of the logic.

**To use:** Runtime → Run all. Minutes, not seconds, if you pick the US universe
— see §2.


## 1 · Setup


In [ ]:
# THE REPOSITORY IS PRIVATE. This clone will fail in Colab unless you supply a
# token, and the `2>/dev/null` below will hide the reason from you.
#
# Either paste a token into the URL (https://<TOKEN>@github.com/...), or use
# notebooks/momentum_ranker_colab.ipynb instead — that one carries its own code
# and needs no clone at all.
!git clone --depth 1 https://github.com/tli9181991/qqq_boxx_strategies.git || git -C qqq_boxx_strategies pull --ff-only
!pip install -q "yfinance>=0.2.40" "lxml>=4.9" "finvizfinance>=1.0"

import os, sys
REPO = "/content/qqq_boxx_strategies"
if not os.path.isdir(REPO):          # running outside Colab, e.g. locally
    REPO = os.getcwd()
os.chdir(REPO)                       # the caches live at data/, resolved relatively
sys.path.insert(0, REPO)

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
print("repo at", REPO, "@", os.popen("git rev-parse --short HEAD").read().strip())


## 2 · Choose the universe

**This is the decision that matters in this notebook.**

| | `"ndx"` | `"us"` |
|---|---|---|
| Names | ~99 Nasdaq-100 | ~2,400 US common + ADR |
| Time | under a minute | **several minutes** |
| Source | Wikipedia + yfinance | Finviz screener + yfinance |
| A top-20 is… | usually *everyone who qualified* | a real selection |

On the Nasdaq-100 a **median of 11** names clear the filter, so asking for 20
mostly returns the whole qualifying set rather than the strongest 20 of a wide
field. That is not a bug — it is what an absolute bar does to a 99-name
universe — but it does mean the ranking is barely doing any work.

The Finviz screener paginates at 20 rows a page, so ~2,400 names is ~120
requests with a polite pause between them. Both paths cache to `data/`, so a
second run in the same session is fast.


In [ ]:
UNIVERSE = "ndx"        # "ndx" (fast) or "us" (the real thing, minutes)
N_HOLD   = 20           # how many names to rank down to
START    = "2022-01-01" # enough history for the 252-day RS lookback

from dataclasses import replace
from qbs.config import FinvizScreenParams

screen = replace(FinvizScreenParams(), n_hold=N_HOLD)
print(f"close > ${screen.min_price:,.0f} · "
      f"volume > {screen.min_volume:,.0f} shares/day · "
      f"quarterly gain > {screen.min_quarter_return:.0%} over {screen.quarter_lookback} sessions")
print(f"then RS-ranked on {screen.rs_lookback} bars, top {screen.n_hold} kept")


## 3 · Data

Closes **and share volumes** — the screen refuses to run its 300k-share leg
without volume rather than skipping it silently, so a screen that ran is a
screen that applied its whole definition.

The safe asset (BOXX) is downloaded separately: the universe caches hold index
members only, and asking them for BOXX would quietly drop the cash leg.


In [ ]:
from qbs.data import load_prices

if UNIVERSE == "us":
    from qbs.finviz import UniverseFilters, fetch_us_universe, load_universe_bars

    filters = UniverseFilters()
    uni_table, err = fetch_us_universe(filters, verbose=True)
    if uni_table is None:
        raise RuntimeError(f"Finviz screener failed: {err}")
    tickers = uni_table["Ticker"].tolist()
    print(f"{len(tickers)} names from Finviz · {filters.label}")

    closes, volumes, err = load_universe_bars(tickers, start=START, verbose=True)
    if closes is None or closes.empty:
        raise RuntimeError(f"price download failed: {err}")
else:
    from qbs.universe import (load_universe, load_universe_prices,
                              load_universe_volumes)

    tickers = load_universe(fetch=True, warn=False)
    closes = load_universe_prices(tickers, start=START, refresh=True)
    volumes = load_universe_volumes(list(closes.columns))

safe = load_prices(["BOXX"], start=START, refresh=True)["BOXX"]

if volumes is None or volumes.empty:
    raise RuntimeError(
        "No share volumes were cached, so the screen's 300k-share leg "
        "cannot run. Re-run this cell with refresh=True, or set "
        "screen = replace(screen, min_volume=None) to run WITHOUT that leg "
        "-- which makes the screen more permissive than its definition.")

volumes = volumes.reindex(index=closes.index).ffill()
safe = safe.reindex(closes.index).ffill()
print(f"{closes.shape[1]} names · {closes.shape[0]} sessions · last bar {closes.index.max():%Y-%m-%d}")


## 4 · Run the screen


In [ ]:
from qbs.screens import finviz_momentum_screen

sig = finviz_momentum_screen(closes, safe, screen, volumes=volumes)

asof = max(sig.holdings_log)
picks = list(sig.holdings_log[asof])       # already in RS order
print(f"{asof:%Y-%m-%d} — {len(picks)} of {screen.n_hold} slots filled")
if len(picks) < screen.n_hold:
    print("   fewer names cleared the filter than the screen ranks down to, "
          "so this is everyone who qualified rather than the strongest N.")
print()
print(", ".join(picks) if picks else "nothing qualified -- the book is in cash")


## 5 · The picks, with the numbers that chose them

Every column here is recomputed from the same frames the screen saw, so the
table explains the selection rather than restating it.

**On `Rank` when few names qualify:** RS Rank buckets the 1-year return into
`min(100, n)` buckets, so with a handful of candidates every name lands in its
own bucket and the order is simply the 1-year return. The tie-break — smallest
distance below the 52-week high — only starts doing work on a wide universe
where names share a bucket.


In [ ]:
import numpy as np

px, vol = closes.loc[:asof], volumes.loc[:asof]
last, lastv = px.iloc[-1], vol.iloc[-1]

perf_1y = px.iloc[-1] / px.iloc[-1 - screen.rs_lookback] - 1.0
qtr = px.iloc[-1] / px.iloc[-1 - screen.quarter_lookback] - 1.0
high = px.tail(screen.high_window).max()
off_high = 1.0 - last / high

table = pd.DataFrame({
    "Close": last, "Volume": lastv, "$ volume": last * lastv,
    "Quarter": qtr, "1-year": perf_1y, "Off high": off_high,
}).loc[picks]
table.insert(0, "Rank", range(1, len(table) + 1))

display(table.style.format({
    "Close": "${:,.2f}", "Volume": "{:,.0f}", "$ volume": "${:,.0f}",
    "Quarter": "{:+.1%}", "1-year": "{:+.1%}", "Off high": "{:.1%}"})
    .background_gradient(subset=["Quarter", "1-year"], cmap="Greens"))

print("Rank is the screen's own order: RS Rank on the 1-year return, "
      "tie-broken by the smallest distance below the 52-week high.")


## 6 · Where everyone else was lost

The legs are independent tests, so the counts below do not add up to the
universe — a name can fail two at once. What they show is which leg is doing
the work, which is usually the quarterly gain.


In [ ]:
legs = {
f"close > ${screen.min_price:,.0f}": last > screen.min_price,
f"volume > {screen.min_volume:,.0f}": lastv > screen.min_volume,
f"quarter > {screen.min_quarter_return:.0%}": qtr > screen.min_quarter_return,
}
rows = [{"Leg": k, "Pass": int(v.sum()),
         "Share": v.sum() / len(v)} for k, v in legs.items()]
all_three = np.logical_and.reduce(list(legs.values()))
rows.append({"Leg": "ALL THREE", "Pass": int(all_three.sum()),
             "Share": all_three.sum() / len(all_three)})

funnel = pd.DataFrame(rows)
display(funnel.style.format({"Share": "{:.1%}"}).hide(axis="index"))

qualified = sorted(px.columns[all_three])
print(f"{len(qualified)} qualified, {len(picks)} kept:")
cut = [t for t in qualified if t not in picks]
print("  cut by the ranking:", ", ".join(cut) if cut else "none -- every qualifying name fitted in the book")


## 7 · Take it with you


In [ ]:
out = table.copy()
out.index.name = "Ticker"
fname = f"high_momentum_{UNIVERSE}_{asof:%Y%m%d}.csv"
out.to_csv(fname, encoding="utf-8-sig")
print("wrote", fname)

try:                       # Colab only
    from google.colab import files
    files.download(fname)
except Exception:
    print("(not on Colab -- the file is in the working directory)")


---

### Reading this honestly

* **This is a screen, not a backtest.** It says what qualifies today. It says
  nothing about whether holding these names makes money — for that, run
  `pipeline.run()` and read the summary table.
* **The universe is survivorship-biased** on the `"ndx"` path: today's index
  membership applied to all history. It flatters any cross-sectional result.
* **Market cap is not filtered.** The original Finviz screen wanted over $300m,
  which needs fundamentals this package does not carry. On the `"us"` universe
  that omission is permissive.
* **A quiet session can drop a name.** The volume leg reads the single most
  recent session, not an average, so a holiday half-day can push a liquid name
  under 300k shares for a day.
